## ColBERT

ColBERT is a **neural information retrieval model** that query ra document lai token-level ma compare garera relevant documents retrieve garcha.

It provides more detailed matching than traditional vector similarity.


In [1]:
from pylate import models

# Pretrained ColBERT v2 model load gareko
model = models.ColBERT(model_name_or_path="colbert-ir/colbertv2.0")

c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name colbert-ir/colbertv2.0.
c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--colbert-ir--colbertv2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support 

In [2]:
import requests


def get_wikipedia_page(title: str):
    """
    Wikipedia bata page ko full text content load garne function.
    """

    # Wikipedia API ko URL
    URL = "https://en.wikipedia.org/w/api.php"

    # API lai pathaune parameters
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }

    # Wikipedia lai request ko purpose identify garna User-Agent diyeko
    headers = {"User-Agent": "RAGatouille_tutorial/0.0.1"}

    # Wikipedia API ma request pathaeko
    response = requests.get(URL, params=params, headers=headers)

    # Response lai JSON ma convert gareko
    data = response.json()

    # Wikipedia bata page extract gareko
    page = next(iter(data["query"]["pages"].values()))

    # Page ko text return gareko
    return page.get("extract")


# Hayao Miyazaki ko Wikipedia page load gareko
full_document = get_wikipedia_page("Hayao_Miyazaki")

print(full_document[:1000])

Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is widely regarded as one of the greatest and most accomplished filmmakers in the history of animation.
Born in Tokyo City, Miyazaki expressed interest in manga and animation from an early age. He joined Toei Animation in 1963, working as an inbetween artist and key animator on films including Gulliver's Travels Beyond the Moon (1965), The Great Adventure of Horus, Prince of the Sun (1968), and Animal Treasure Island (1971), before moving to A-Pro in 1971, where he co-directed Lupin the Third Part I (1971–1972) alongside Isao Takahata. After moving to Zuiyō Eizō (later Nippon Animation) in 1973, Miyazaki worked as an animator on World Master

In [5]:
from pylate import indexes

# Wikipedia bata aayeko document lai chunks ma divide garne
# RAGatouille ko max_document_length=180 jastai directly option chaina,
# tesley pahila document lai chunks ma split garne

chunks = [full_document[i : i + 180] for i in range(0, len(full_document), 180)]

# Chunks ko ColBERT embeddings generate gareko
document_embeddings = model.encode(
    chunks,
    is_query=False,
    show_progress_bar=True,
)

# PLAID index create gareko
index = indexes.PLAID(
    index_folder="colbert-index",
    index_name="Miyazaki-123",
    override=True,
)

# Chunks lai index ma store gareko
index.add_documents(
    documents_ids=[str(i) for i in range(len(chunks))],
    documents_embeddings=document_embeddings,
)

print(f"Indexed {len(chunks)} chunks")

Encoding documents (bs=32):   0%|          | 0/13 [00:00<?, ?it/s]

Encoding documents (bs=32): 100%|██████████| 13/13 [00:17<00:00,  1.36s/it]


Indexed 392 chunks


In [6]:
from pylate import retrieve

# ColBERT retriever create gareko
retriever = retrieve.ColBERT(index=index)

# User ko query
query = "What animation studio did Miyazaki found?"

# Query ko ColBERT embedding generate gareko
query_embedding = model.encode(
    [query],
    is_query=True,
    show_progress_bar=True,
)

# Top 3 relevant chunks retrieve gareko
results = retriever.retrieve(
    queries_embeddings=query_embedding,
    k=3,
)

results

Encoding queries (bs=32): 100%|██████████| 1/1 [00:00<00:00, 13.20it/s]


[[{'id': '130', 'score': 25.37744140625},
  {'id': '118', 'score': 24.902099609375},
  {'id': '79', 'score': 24.798828125}]]

In [7]:
from pylate import retrieve

# ColBERT retriever create gareko
retriever = retrieve.ColBERT(index=index)

# User ko question
query = "What animation studio did Miyazaki found?"

# Query ko ColBERT embedding generate gareko
query_embedding = model.encode(
    [query],
    is_query=True,
    show_progress_bar=True,
)

# Top 3 relevant chunks retrieve gareko
results = retriever.retrieve(
    queries_embeddings=query_embedding,
    k=3,
)

results

Encoding queries (bs=32): 100%|██████████| 1/1 [00:00<00:00, 13.04it/s]


[[{'id': '130', 'score': 25.37744140625},
  {'id': '118', 'score': 24.902099609375},
  {'id': '79', 'score': 24.798828125}]]